# 04 — Extract Calibrated Spectrum (Routine Use)

**Purpose:** Load a CMOS image, apply the existing pattern and wavelength calibration, 
extract a calibrated spectrum, and optionally save it.

**When to run:** This is the standard daily workflow for plasma experiments. 
No recalibration required — just point to your image file.

**Output:**
- `Spectrum` object with wavelength in nm and intensity in W/(m² nm), W/(m² sr nm), Nph/(m² sr)
- Optional: `SHOTNAME_echelle_spec.txt` text file

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from echelle_spectra.tools.echelle import Calibrations, EchelleImage, Spectrum
from echelle_spectra.tools import emissiondata as ebd
from echelle_spectra.tools.emissionbands import banddata

import echelle_spectra
CALIB_DIR = echelle_spectra._config['base_path'] / 'resources/calibration_files'

%matplotlib inline

## Configuration

The only thing you need to change for each new image is `IMAGE_PATH`.

In [ ]:
# ------------ CHANGE THIS FOR EACH SESSION ------------
IMAGE_PATH  = "/path/to/your/shot_XXXXX.sif"
OUTPUT_DIR  = "/path/to/output/"  # where to save the extracted spectrum text file
SHOT_NUMBER = None                # set to an int or string, or leave None to use filename
# -------------------------------------------------------

# Current CMOS calibration files (change only if recalibrated)
FILES_CMOS = {
    "orders":    "pattern_CMOS_20240305.txt",
    "wavelength": "Th_wavelength_CMOS_20240305.txt",
    "sphr":      "sphere_cmos_20240305.sif",
    "bkgr":      "sphere_cmos_20240305_bkg.sif",
    "integral":  "integrating_sphere.txt",
}

SAVE_SPECTRUM = False  # set True to write output text file
SAVE_UNITS    = "wm"  # 'counts', 'wm', 'wmsr', or 'phmsr'

## Load Calibration

This loads the pattern, wavelength solution, integrating sphere, and computes absolute sensitivity. 
Takes ~10–30 s the first time; subsequent runs in the same session are fast.

In [ ]:
cb = Calibrations(folder=str(CALIB_DIR), filenames=FILES_CMOS)
cb.start()
print("Calibration loaded.")
print(f"  Orders:    {cb.pattern.shape[1]}")
print(f"  Detector:  {cb.DIMO} rows × {cb.DIMW} cols")
print(f"  Wavelength range: {cb.wavelength.min():.1f} – {cb.wavelength.max():.1f} nm")

## Load Image and Extract Spectrum

In [ ]:
em = EchelleImage(IMAGE_PATH, clbr=cb)
em.calibrate()   # extracts orders, corrects shapes, stitches

sp = Spectrum(em)

print(f"Frames:    {sp.counts.shape[0]}")
print(f"Pixels:    {sp.counts.shape[1]}")
print(f"Exposure:  {sp.info['ExposureTime']} s")
if sp.info['BackgroundFrames']:
    print(f"Background frames auto-detected: {sp.info['BackgroundFrames']}")

## Plot Full Spectrum

In [ ]:
frame = 0  # which frame to plot

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(sp.wavelength, sp.counts[frame], lw=0.8, color='k')
axes[0].set_ylabel("Counts (background-subtracted)")
axes[0].set_title(f"Echelle spectrum — frame {frame}")

axes[1].plot(sp.wavelength, sp.wm[frame], lw=0.8, color='tab:red')
axes[1].set_ylabel("W / (m² nm)")
axes[1].set_xlabel("Wavelength (nm)")

plt.tight_layout()
plt.show()

## Plot Raw 2D Image with Order Overlay

In [ ]:
import matplotlib.colors as mcolors

img = em.images[0]
norm = mcolors.LogNorm(vmin=max(img.min(), 1), vmax=img.max())

fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(img, origin='lower', cmap='inferno', norm=norm, aspect='auto')

# Overlay order pattern
n_cols = cb.pattern.shape[0]
for j in range(cb.pattern.shape[1]):
    ax.plot(np.arange(n_cols), cb.pattern[:, j], 'w-', lw=0.5, alpha=0.5)

ax.set_title("Raw image with order pattern overlay")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.tight_layout()
plt.show()

## Emission Band Analysis: Hydrogen Fulcher-alpha

Extract the H₂ Fulcher-α band and fit Gaussian emission line profiles.

In [ ]:
# Use the predefined Balmer Hα band from emissiondata
# Other available bands: hbeta, hgamma, hdelta, he587, he667, c444, c547, ...
band = ebd.halpha

# Extract the wavelength/intensity slice for this band
wl_band, int_band = banddata(sp, frame=frame, band=band)

plt.figure(figsize=(8, 4))
plt.plot(wl_band, int_band, 'k-', lw=0.8, label='data')
plt.xlabel("Wavelength (nm)")
plt.ylabel("W / (m² nm)")
plt.title(f"{band.name}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gaussian fit of the emission band
fit_result = band.fitb(frame=frame, sp=sp)
print(fit_result.report())

## Save Spectrum to Text File

In [ ]:
if SAVE_SPECTRUM:
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    sp.output_path = OUTPUT_DIR
    sp.saveunits    = SAVE_UNITS
    if SHOT_NUMBER is not None:
        sp.shotnumber = SHOT_NUMBER
    sp.save()
    print(f"Spectrum saved to: {OUTPUT_DIR}")
else:
    print("SAVE_SPECTRUM=False — not writing file.")
    print("Set SAVE_SPECTRUM=True and configure OUTPUT_DIR to save.")

---
## Quick Reference: Available Emission Bands

```python
from echelle_spectra.tools import emissiondata as ebd

# Carbon ions
ebd.c444, ebd.c465, ebd.c547, ebd.c580, ebd.c706, ebd.c772  # CIV
ebd.c515, ebd.c464                                            # CII/CIII

# Helium
ebd.he447, ebd.he492, ebd.he501, ebd.he587, ebd.he667, ebd.he706

# Hydrogen Balmer
ebd.halpha, ebd.hbeta, ebd.hgamma, ebd.hdelta

# Xenon
ebd.xe461, ebd.xe462, ebd.xe467
```